In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --------------------------------------------------
# STEP 1: Read source data from Silver
# --------------------------------------------------

df_src = spark.sql("""
    SELECT DISTINCT Branch_ID, BranchName
    FROM parquet.`abfss://carsales@adls7786.dfs.core.windows.net/silver`
""")

# --------------------------------------------------
# STEP 2: Check whether Gold dimension exists
# --------------------------------------------------

table_name = "car_sales.gold.dim_branch"

if spark.catalog.tableExists(table_name):

    df_sink = spark.sql("""
        SELECT dim_branch_key, Branch_ID, BranchName
        FROM car_sales.gold.dim_branch
    """)

    max_value = (
        spark.sql("""
            SELECT COALESCE(MAX(dim_branch_key), 0)
            FROM car_sales.gold.dim_branch
        """).collect()[0][0]
    )

else:

    df_sink = (
        df_src.limit(0)
        .withColumn(
            "dim_branch_key",
            F.lit(None).cast("long")
        )
        .select("dim_branch_key", "Branch_ID", "BranchName")
    )

    max_value = 0

# --------------------------------------------------
# STEP 3: Identify existing and new branches
# --------------------------------------------------

df_filter = (
    df_src.alias("src")
    .join(
        df_sink.alias("sink"),
        F.col("src.Branch_ID") == F.col("sink.Branch_ID"),
        "left"
    )
    .select(
        F.col("src.Branch_ID").alias("Branch_ID"),
        F.col("src.BranchName").alias("BranchName"),
        F.col("sink.dim_branch_key").alias("dim_branch_key")
    )
)

# Existing branches
df_filter_old = df_filter.filter(
    F.col("dim_branch_key").isNotNull()
)

# New branches
df_filter_new = (
    df_filter.filter(F.col("dim_branch_key").isNull())
    .select("Branch_ID", "BranchName")
)

# --------------------------------------------------
# STEP 4: Generate surrogate keys for new branches
# --------------------------------------------------

window_spec = Window.orderBy("Branch_ID")

df_filter_new = df_filter_new.withColumn(
    "dim_branch_key",
    F.lit(max_value) + F.row_number().over(window_spec)
)

# --------------------------------------------------
# STEP 5: Combine existing and new records
# --------------------------------------------------

df_final = df_filter_old.unionByName(df_filter_new)

# --------------------------------------------------
# STEP 6: MERGE into Gold dimension
# --------------------------------------------------

if spark.catalog.tableExists(table_name):

    delta_table = DeltaTable.forPath(
        spark,
        "abfss://carsales@adls7786.dfs.core.windows.net/gold/dim_branch"
    )

    delta_table.alias("t").merge(
        df_final.alias("s"),
        "t.Branch_ID = s.Branch_ID"
    ).whenMatchedUpdate().whenNotMatchedInsertAll().execute()

else:

    (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .option(
            "path",
            "abfss://carsales@adls7786.dfs.core.windows.net/gold/dim_branch"
        )
        .saveAsTable(table_name)
    )
